# Fase 3: Modelagem e Experimentos

Este notebook executa o pipeline de experimentação para previsão de risco criminal no entorno do IFSP-SPO, atendendo aos passos 1 a 7 do protocolo experimental da disciplina e os requisitos do artigo (comparação de 4 modelos).

In [ ]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

# Garantir que os diretórios de salvamento existam
os.makedirs("../models/checkpoints", exist_ok=True)
os.makedirs("../models/final", exist_ok=True)

sns.set_theme(style="whitegrid")

## 1. Carga e Preparação das Features
Vamos carregar a base unificada com a variável alvo (RISCO_ALTO) e extrair as features necessárias, aplicando One-Hot Encoding.

In [ ]:
# Carregando dataset processado (substitua o caminho se necessário)
df = pd.read_csv("../data/processed/dados_processados.csv", sep=";")

# Garantir que DATA é datetime e extrair variáveis temporais
df["DATA"] = pd.to_datetime(df["DATA"])
df["MES"] = df["DATA"].dt.month
df["DIA_SEMANA"] = df["DATA"].dt.dayofweek
df["FLAG_FIM_DE_SEMANA"] = df["DIA_SEMANA"].isin([5, 6]).astype(int)

# Ordenar cronologicamente para garantir separação temporal correta
df = df.sort_values(["DATA"]).reset_index(drop=True)

# Definir matriz de Features (X) e Target (y)
features = ["MES", "DIA_SEMANA", "FLAG_FIM_DE_SEMANA"] + [
    col for col in df.columns if "PERIODO_" in col
]
X = df[features]
y = df["RISCO_ALTO"]

X.head()

## 2. Separação Estrita do Conjunto de Teste
Separamos os ~15% finais dos dados chronologicamente para uso EXCLUSIVO no Passo 6 (avaliação de generalização).

In [ ]:
# Partição cronológica (Sem usar train_test_split com shuffle)
split_idx = int(len(X) * 0.85)

X_dev, y_dev = (
    X.iloc[:split_idx],
    y.iloc[:split_idx],
)  # Desenvolvimento (Treino/Validação)
X_test, y_test = X.iloc[split_idx:], y.iloc[split_idx:]  # Teste Final

print(f"Tamanho Desenvolvimento: {len(X_dev)} instâncias")
print(f"Tamanho Teste Final: {len(X_test)} instâncias")

## Passo 1 do Enunciado: Teste de Implementação (Sanity Check)
Verificação de consistência forçando overfitting num pequeno subconjunto para atestar a capacidade de convergência do modelo.

In [ ]:
# Isolar 30 linhas
X_sanity, y_sanity = X_dev.iloc[:30], y_dev.iloc[:30]

# Treinar modelo não linear (Árvore de Decisão sem limite de profundidade)
modelo_sanity = DecisionTreeClassifier(random_state=42)
modelo_sanity.fit(X_sanity, y_sanity)

# Prever e calcular acurácia no próprio conjunto
y_pred_sanity = modelo_sanity.predict(X_sanity)
acc_sanity = accuracy_score(y_sanity, y_pred_sanity)

print(
    f"Acurácia no Teste de Sanidade (Overfitting proposital): {acc_sanity * 100:.2f}%"
)
assert acc_sanity > 0.95, (
    f"Falha no teste de sanidade: Modelo não conseguiu decorar a amostra. Acurácia: {acc_sanity * 100:.2f}%"
)
print("Implementação testada e validada.")

## Passos 3 e 4: Experimentação e Replicação (4 Modelos)
Realizando a comparação entre 4 algoritmos clássicos para dados tabulares, executando iterações com particionamentos aleatórios estratificados.

In [ ]:
n_iteracoes = 30
resultados = {"Modelo": [], "Seed": [], "F1_Score": [], "Acuracia": []}

# Definição base dos modelos para instanciação
modelos_dict = {
    "Logistic Regression": lambda s: LogisticRegression(random_state=s, max_iter=500),
    "Decision Tree": lambda s: DecisionTreeClassifier(random_state=s, max_depth=6),
    "Random Forest": lambda s: RandomForestClassifier(
        random_state=s, n_estimators=100, max_depth=6
    ),
    "Gradient Boosting": lambda s: GradientBoostingClassifier(
        random_state=s, n_estimators=100, max_depth=3
    ),
}

for seed in range(42, 42 + n_iteracoes):
    # Validação Cruzada Monte Carlo
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev, y_dev, test_size=0.20, random_state=seed, stratify=y_dev
    )

    for nome_modelo, construtor in modelos_dict.items():
        # Instancia e Treina
        modelo = construtor(seed)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_val)

        # Coleta Métricas
        resultados["Modelo"].append(nome_modelo)
        resultados["Seed"].append(seed)
        resultados["F1_Score"].append(f1_score(y_val, pred))
        resultados["Acuracia"].append(accuracy_score(y_val, pred))

        # Salva o checkpoint (Substituir espaços por underscore no nome)
        nome_arquivo = nome_modelo.lower().replace(" ", "_")
        joblib.dump(modelo, f"../models/checkpoints/{nome_arquivo}seed{seed}.pkl")

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("../models/checkpoints/log_experimentos.csv", index=False)
print(
    "Experimentos com 4 modelos executados e resultados/pesos intermediários salvos com sucesso."
)

## Passo 5: Análise Estatística dos Resultados
Comparando as métricas de 4 grupos através do Teste de Friedman (Omnibus), seguido da seleção do melhor.

In [ ]:
# Tabela descritiva
display(
    df_resultados.groupby("Modelo")
    .agg({"F1_Score": ["mean", "std", "median"], "Acuracia": ["mean", "std"]})
    .sort_values(by=("F1_Score", "mean"), ascending=False)
)

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_resultados, x="Modelo", y="F1_Score", palette="Set2", ax=ax)
ax.set_title(
    "Distribuição do F1-Score nas Múltiplas Partições de Validação (4 Modelos)",
    weight="bold",
)
plt.show()

# Preparar arrays de F1-Score por modelo
f1_lr = df_resultados[df_resultados["Modelo"] == "Logistic Regression"][
    "F1_Score"
].values
f1_dt = df_resultados[df_resultados["Modelo"] == "Decision Tree"]["F1_Score"].values
f1_rf = df_resultados[df_resultados["Modelo"] == "Random Forest"]["F1_Score"].values
f1_gb = df_resultados[df_resultados["Modelo"] == "Gradient Boosting"]["F1_Score"].values

# Teste de Friedman (Omnibus não-paramétrico para >2 grupos repetidos)
stat_friedman, p_friedman = stats.friedmanchisquare(f1_lr, f1_dt, f1_rf, f1_gb)
print(f"Teste de Friedman (p-valor): {p_friedman:.4e}")

melhor_modelo = df_resultados.groupby("Modelo")["F1_Score"].mean().idxmax()

if p_friedman < 0.05:
    print("Há significância estatística de que pelo menos um modelo difere dos demais.")
    print(f"Selecionando o modelo com maior média global: {melhor_modelo}")
else:
    melhor_modelo = "Logistic Regression"  # Baseline
    print(
        "Não há diferença estatística global (p > 0.05). Adotaremos o baseline (Regressão Logística)."
    )


## Passo 6: Estimativa de Erro em Dados Não Vistos (Generalização)
Treinaremos o modelo vencedor em todo o conjunto de desenvolvimento e mediremos no X_test.

In [ ]:
# Instancia modelo vencedor usando o dicionário definido anteriormente com semente fixa
modelo_val_final = modelos_dict[melhor_modelo](42)

# Treina no conjunto Dev inteiro
modelo_val_final.fit(X_dev, y_dev)

# Prever e avaliar no conjunto Teste Isolado
y_test_pred = modelo_val_final.predict(X_test)

print("=== RELATÓRIO FINAL DE GENERALIZAÇÃO ===")
print(f"Modelo Selecionado: {melhor_modelo}")
print(classification_report(y_test, y_test_pred))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Matriz de Confusão ({melhor_modelo} - Teste)")
ax.set_xlabel("Previsto")
ax.set_ylabel("Real")
plt.show()

## Passo 7: Treinamento da Versão Final
Treinamento do artefato final com todos os dados disponíveis (Dev + Test) para entrega/deploy.

In [ ]:
# Instancia modelo vencedor para deploy
modelo_deploy = modelos_dict[melhor_modelo](42)

# Treina em TODO o dataset (X e y)
modelo_deploy.fit(X, y)

# Salva o arquivo final
caminho_final = "../models/final/modelo_risco_ifsp_final.joblib"
joblib.dump(modelo_deploy, caminho_final)

print(f"Artefato final treinado e salvo com sucesso em: {caminho_final}")